# Checkpoint 3: External evaluation of ICU mortality prediction

## AI Use Statement

**How AI Was Used**

I used ChatGPT and OpenAI Codex to assist with this project. Codex helped me set up the programming environment, write Python code, and debug errors. ChatGPT helped me understand the assignment requirements, plan the workflow, and answer questions when I did not understand something. For example, AI helped me solve Pixi environment issues during Checkpoint 2,3 and set up my GitHub repository.

**Pluses of Using AI**

AI was very convenient and helped reduce the time I spent on coding. For me, the most useful part was the assistance with writing code.

**Minuses of Using AI**

One limitation was that I did not fully understand some of the AI-assisted code. Therefore, I needed to spend additional time studying the code to understand how it worked.

In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess
from sklearn.metrics import roc_auc_score, precision_recall_curve, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from IPython.display import display
from checkpoint3_support import build_features
SEED = 707
rng = np.random.default_rng(SEED)
OUT = Path('output')
meta = json.loads((OUT/'cp2_model_metadata.json').read_text(encoding='utf-8'))
model = joblib.load(OUT/'cp2_final_model.joblib')
predictors = meta['predictors']
comparison = pd.read_csv(OUT/'cp2_comparison.csv', index_col=0)
oof = pd.read_csv(OUT/'cp2_oof_predictions.csv')
print('Saved model:', meta['selected_model'], '| ordered predictor count:', len(predictors))
print('Pipeline:', model)


## 3.1 Apply the final model to Sets B and C

The Checkpoint 2 outcome-independent feature builder and fitted preprocessing pipeline are applied without refitting. Set C contains 2,403 observations with a blank parameter name; their identities cannot be recovered, so these unlabeled rows are excluded before the otherwise unchanged feature builder runs. Set B has no such rows. This source-data limitation may affect Set C results. Gender and ICU type retain the saved pipeline's categorical encoding. All predictor columns must match the saved input names and order. Negative benchmark scores indicate missing values.


In [ ]:
sets = {}
for name in ['b','c']:
    X = build_features(Path('data')/('set-'+name))
    raw = pd.read_csv(Path('data')/('Outcomes-'+name+'.txt'))
    assert X.index.is_unique and raw.RecordID.is_unique
    raw = raw.set_index('RecordID')
    assert set(X.index)==set(raw.index), f'Set {name}: IDs do not match'
    assert set(X.columns)==set(predictors), f'Set {name}: predictor mismatch'
    X = X.loc[:, predictors]
    assert list(X.columns)==predictors
    assert not set(['In-hospital_death','SAPS-I','SOFA','Survival','Length_of_stay']).intersection(X.columns)
    y = raw.loc[X.index,'In-hospital_death']
    assert y.notna().all() and y.isin([0,1]).all()
    risk = model.predict_proba(X)[:,1]
    assert np.isfinite(risk).all() and ((risk>=0)&(risk<=1)).all()
    sets[name] = (X, y, risk, raw.loc[X.index])
cohort = pd.DataFrame([{'Set':k.upper(),'Stays':len(y),'Deaths':int(y.sum()),'Mortality':y.mean()} for k,(X,y,p,z) in sets.items()])
display(cohort)
print('Unique matched RecordIDs, complete binary outcomes and exact ordered model inputs verified for both sets.')


## 3.2 Discrimination

AUROC confidence intervals use 2,000 patient-level bootstrap samples (seed 707); single-class resamples are excluded. Benchmark AUROCs use available scores, with their sample sizes shown. The saved Checkpoint 2 comparison uses only the common complete-score cohort and reports fold mean and sample SD; external results use their available cohorts, so populations differ.


In [ ]:
def auc(y,p):
    return roc_auc_score(y,p) if np.unique(y).size == 2 else np.nan
def auc_ci(y,p):
    y=np.asarray(y); p=np.asarray(p); values=[]
    for _ in range(2000):
        ix=rng.integers(len(y),size=len(y))
        if np.unique(y[ix]).size==2: values.append(auc(y[ix],p[ix]))
    return np.quantile(values,[.025,.975]) if values else [np.nan,np.nan]
rows=[{'Dataset':'Set A: 5-fold CV, complete-score cohort','Measure':meta['selected_model'],'N':meta['benchmark_complete_n'],'AUROC':comparison.loc[meta['selected_model'],'Mean'],'Lower 95%':np.nan,'Upper 95%':np.nan,'Fold SD':comparison.loc[meta['selected_model'],'SD']}]
for k,(X,y,p,z) in sets.items():
    lo,hi=auc_ci(y,p)
    rows.append({'Dataset':'Set '+k.upper(),'Measure':'Final model','N':len(y),'AUROC':auc(y,p),'Lower 95%':lo,'Upper 95%':hi})
    for score in ['SAPS-I','SOFA']:
        s=z[score].where(z[score]>=0); ok=s.notna()
        rows.append({'Dataset':'Set '+k.upper(),'Measure':score,'N':int(ok.sum()),'AUROC':auc(y[ok],s[ok])})
display(pd.DataFrame(rows).round(3))
b_y=sets['b'][1].to_numpy(); b_p=sets['b'][2]; c_y=sets['c'][1].to_numpy(); c_p=sets['c'][2]; differences=[]
for _ in range(2000):
    ib=rng.integers(len(b_y),size=len(b_y)); ic=rng.integers(len(c_y),size=len(c_y))
    if np.unique(b_y[ib]).size==2 and np.unique(c_y[ic]).size==2: differences.append(auc(b_y[ib],b_p[ib])-auc(c_y[ic],c_p[ic]))
dlo,dhi=np.quantile(differences,[.025,.975]); cv=comparison.loc[meta['selected_model'],'Mean']
print(f'Set B minus Set C AUROC: {auc(b_y,b_p)-auc(c_y,c_p):.3f}; bootstrap 95% interval [{dlo:.3f}, {dhi:.3f}]. The interval '+('includes' if dlo<=0<=dhi else 'excludes')+' zero; it does not establish clinical importance.')
print(f'Set A complete-score CV mean {cv:.3f}; external differences from this estimate also reflect cohort differences and uncertainty.')
print('Precision-recall curves emphasize the fraction of flagged patients who die and are informative when deaths are uncommon.')
y=sets['b'][1]; p=sets['b'][2]
precision,recall,_=precision_recall_curve(y,p)
fig,ax=plt.subplots(); ax.plot(recall,precision,label='Final model'); ax.axhline(y.mean(),color='black',ls='--',label=f'Prevalence = {y.mean():.3f}'); ax.set(xlabel='Recall (sensitivity)',ylabel='Precision (PPV)',title='Set B precision-recall curve',xlim=(0,1),ylim=(0,1)); ax.legend(); plt.show()


## 3.3 Calibration

The smooth curves use LOWESS with a pointwise 95% patient-bootstrap band (100 replicates, fixed seed). The rug histogram shows the risk distribution. Calibration-in-the-large is the intercept in a logistic model with logit risk as an offset; the slope comes from a separate logistic model with an estimated intercept. Risks are clipped at 10^-6 before taking logits. Scaled Brier score uses the evaluation-set prevalence as the reference prediction.


In [ ]:
def logit(p):
    p=np.clip(np.asarray(p),1e-6,1-1e-6); return np.log(p/(1-p))
def citl(y,p):
    if np.unique(y).size<2: return np.nan
    try: return sm.GLM(np.asarray(y),np.ones((len(y),1)),family=sm.families.Binomial(),offset=logit(p)).fit().params[0]
    except Exception: return np.nan
def slope(y,p):
    try: return sm.GLM(np.asarray(y),sm.add_constant(logit(p)),family=sm.families.Binomial()).fit().params[1]
    except Exception: return np.nan
def smooth(p,y,grid):
    xy=lowess(np.asarray(y),np.asarray(p),frac=.5,it=0,return_sorted=True)
    return np.interp(grid,xy[:,0],xy[:,1],left=np.nan,right=np.nan)
cal=[]
for k,(X,y,p,z) in sets.items():
    bs=brier_score_loss(y,p); ref=y.mean()*(1-y.mean())
    cal.append({'Set':k.upper(),'Calibration intercept':citl(y,p),'Calibration slope':slope(y,p),'Brier':bs,'Scaled Brier':1-bs/ref if ref else np.nan})
    grid=np.linspace(np.quantile(p,.01),np.quantile(p,.99),100); fit=smooth(p,y,grid); boots=[]
    for _ in range(100):
        ix=rng.integers(len(y),size=len(y)); boots.append(smooth(p[ix],y.to_numpy()[ix],grid))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',RuntimeWarning); lo,hi=np.nanquantile(boots,[.025,.975],axis=0)
    fig,ax=plt.subplots(); ax.plot([0,1],[0,1],'k--',label='Ideal'); ax.plot(grid,fit,label='LOWESS'); ax.fill_between(grid,lo,hi,alpha=.2,label='95% pointwise band'); ax.set(xlabel='Predicted risk',ylabel='Observed mortality',title=f'Set {k.upper()} calibration',xlim=(0,1),ylim=(0,1)); ax.legend(loc='upper left'); ax2=ax.twinx(); ax2.hist(p,bins=25,range=(0,1),alpha=.12,color='gray'); ax2.set_ylabel('Number of stays'); plt.show()
display(pd.DataFrame(cal).round(3))
for row in cal:
    print(f"Set {row['Set']}: intercept {row['Calibration intercept']:.2f} (0 ideal), slope {row['Calibration slope']:.2f} (1 ideal). Examine the smooth curve and histogram together to judge where deviations are supported by data.")


## 3.4 Clinical usefulness

Hypothetical application: flag patients after 48 hours for an additional clinician review at a predicted mortality risk of 20%. This is an illustrative threshold, not an established clinical policy. Decision curves use thresholds from 5% to 50%. A logistic SOFA-to-risk mapping is fitted using Set A outcomes and SOFA only, then applied unchanged to each external set. Strategies are compared on patients with an available SOFA score; curves do not demonstrate deployment readiness.


In [ ]:
a=pd.read_csv(OUT/'cp2_oof_predictions.csv'); av=a['SOFA'].notna() & (a['SOFA']>=0)
sofa_map=LogisticRegression(C=1e6,solver='lbfgs').fit(a.loc[av,['SOFA']],a.loc[av,'In-hospital_death'])
def netbenefit(y,flag,t):
    y=np.asarray(y); flag=np.asarray(flag); return np.mean(flag & (y==1))-np.mean(flag & (y==0))*t/(1-t)
thresholds=np.linspace(.05,.5,100)
for k,(X,y,p,z) in sets.items():
    ok=z.SOFA.notna() & (z.SOFA>=0); yy=y[ok].to_numpy(); pp=p[ok]; sp=sofa_map.predict_proba(z.loc[ok,['SOFA']])[:,1]
    fig,ax=plt.subplots()
    for label,probs in [('Final model',pp),('SOFA mapped on Set A',sp)]: ax.plot(thresholds,[netbenefit(yy,probs>=t,t) for t in thresholds],label=label)
    ax.plot(thresholds,[netbenefit(yy,np.ones(len(yy),dtype=bool),t) for t in thresholds],label='Treat all'); ax.axhline(0,color='black',ls='--',label='Treat none'); ax.set(xlabel='Risk threshold',ylabel='Net benefit per patient',title=f'Set {k.upper()} decision curves (n={len(yy)})'); ax.legend(); plt.show()
    print(f'Set {k.upper()}: {len(y)-len(yy)} stays excluded from DCA because SOFA was missing.')
y=sets['b'][1].to_numpy(); p=sets['b'][2]; flag=p>=.2; tp=np.sum(flag & (y==1)); sensitivity=tp/np.sum(y==1); ppv=tp/np.sum(flag) if flag.sum() else np.nan
display(pd.DataFrame([{'Set':'B','Threshold':.2,'Sensitivity':sensitivity,'PPV':ppv,'Flagged per 100':100*flag.mean(),'Event 1 score':min(sensitivity,ppv) if np.isfinite(ppv) else np.nan}]).round(3))


## 3.5 Subgroup analysis (Set B)

Age groups are under 50, 50-64, 65-79, and 80 years or older. Gender and ICU type use the recorded source codes. Estimates are omitted for single-class groups; small groups and estimates based on fewer than 10 deaths should be interpreted cautiously. Calibration intercept is unadjusted within each subgroup.


In [ ]:
X,y,p,z=sets['b']
groups={'Age group':pd.cut(X.Age,[-np.inf,50,65,80,np.inf],right=False,labels=['<50','50-64','65-79','80+']).astype('object').fillna('Missing'), 'Sex (Gender code)':X.Gender.fillna(-1).astype(str), 'ICU type code':X.ICUType.fillna(-1).astype(str)}
rows=[]
for var,g in groups.items():
    for label in sorted(g.unique(),key=str):
        mask=np.asarray(g==label); yy=y.to_numpy()[mask]; pp=p[mask]; deaths=int(yy.sum())
        rows.append({'Group':var,'Level':label,'N':len(yy),'Deaths':deaths,'AUROC':auc(yy,pp),'Calibration intercept':citl(yy,pp),'Caution':'<10 deaths or survivors' if min(deaths,len(yy)-deaths)<10 else ''})
display(pd.DataFrame(rows).round(3))
print('Subgroup differences are descriptive; uneven subgroup sizes, few deaths and multiple comparisons limit the strength of comparisons. [AUTHOR: judge which differences matter clinically.]')


## 3.6 Model interpretation

Permutation importance measures the decrease in Set B AUROC when one input is shuffled; correlated inputs can substitute for each other. One repeat on a reproducible 800-stay sample limits runtime, so estimates are noisy. The age partial dependence averages predictions after setting age to each value for the same sample. The Checkpoint 2 Model 2 curve is a reference-profile effect on log odds, not an average risk, so the scales and estimands differ. Neither association nor importance establishes a causal effect. For example, measurement counts can reflect monitoring intensity and clinical decisions as well as physiology; changing monitoring frequency cannot be assumed to change mortality as the model association might suggest.


In [ ]:
X,y,p,z=sets['b']; ix=rng.choice(len(y),size=min(800,len(y)),replace=False); sample=X.iloc[ix]; sy=y.iloc[ix]
perm=permutation_importance(model,sample,sy,scoring='roc_auc',n_repeats=1,random_state=SEED,n_jobs=1)
imp=pd.Series(perm.importances_mean,index=X.columns).sort_values(ascending=False).head(15).sort_values()
fig,ax=plt.subplots(figsize=(8,6)); imp.plot.barh(ax=ax); ax.set(xlabel='Decrease in AUROC after permutation',title='Set B permutation importance'); plt.tight_layout(); plt.show()
eff=pd.read_csv(OUT/'cp2_partial_effects.csv'); candidates=['Age','GCS_min','BUN_last']; target=max(candidates,key=lambda v: perm.importances_mean[list(X.columns).index(v)]); age=eff[eff.Predictor==target]; grid=np.linspace(sample[target].quantile(.05),sample[target].quantile(.95),30); pred=[]
for value in grid:
    altered=sample.copy(); altered[target]=value; pred.append(model.predict_proba(altered)[:,1].mean())
fig,ax=plt.subplots(1,2,figsize=(12,4)); ax[0].plot(grid,pred); ax[0].set(xlabel=target,ylabel='Mean predicted risk',title='Final model: partial dependence'); ax[1].plot(age.Value,age.Effect); ax[1].fill_between(age.Value,age.Lower,age.Upper,alpha=.2); ax[1].set(xlabel=target,ylabel='Log odds relative to reference',title='Checkpoint 2 Model 2 partial effect'); plt.tight_layout(); plt.show()


## 3.7 Most confident mistakes (Set B)

The tables show the five highest-risk survivors and five lowest-risk deaths. Missingness counts refer to input predictors and can count multiple summaries of the same measurement. Extreme values and missing measurements are descriptive; they do not establish why an individual prediction was wrong. The following exploratory one-feature perturbations describe sensitivity to selected inputs; they are not causal or complete explanations.


In [ ]:
X,y,p,z=sets['b']; errors=X[['Age','Gender','ICUType','GCS_min','BUN_last','HR_max']].copy(); errors.insert(0,'Observed death',y); errors.insert(0,'Predicted risk',p); errors['Missing input count']=X.isna().sum(axis=1); errors.index.name='RecordID'
print('Highest predicted risks among survivors'); display(errors.loc[y==0].sort_values('Predicted risk',ascending=False).head(5).round(3))
print('Lowest predicted risks among deaths'); display(errors.loc[y==1].sort_values('Predicted risk').head(5).round(3))

# Local one-feature perturbations to Set A medians/modes; these are not additive attributions.
train=pd.read_csv(OUT/'cp2_modeling_table.csv'); reference={v:(train[v].mode().iloc[0] if v in ['Gender','ICUType'] else train[v].median()) for v in imp.index}
selected=pd.concat([errors.loc[y==0].sort_values('Predicted risk',ascending=False).head(5),errors.loc[y==1].sort_values('Predicted risk').head(5)])
local=[]
for rid in selected.index:
    original=X.loc[[rid]]; baseline=float(model.predict_proba(original)[0,1]); changes=[]
    for v in imp.index:
        altered=original.copy(); altered.loc[rid,v]=reference[v]; delta=baseline-float(model.predict_proba(altered)[0,1]); changes.append((v,delta))
    changes=sorted(changes,key=lambda x:abs(x[1]),reverse=True)[:3]
    local.append({'RecordID':rid,'Largest risk changes when replacing one input with Set A reference':'; '.join(f'{v}: {d:+.3f}' for v,d in changes)})
display(pd.DataFrame(local))
print('Positive changes indicate that the original value raised risk relative to its reference replacement; correlated summaries and unrealistic combinations limit interpretation. [AUTHOR: review each case and judge whether available data support a data-quality or model limitation.]')


## 3.8 Written conclusions

**Clinical Usefulness**

The final gradient boosting model showed good discrimination in both Set B (AUROC = 0.864) and Set C (AUROC = 0.866), outperforming SAPS-I and SOFA. The calibration intercepts were close to zero, although the slopes were slightly above one, suggesting some miscalibration. Decision curve analysis showed a higher net benefit for the final model than the comparison strategies in Set B. At a 20% risk threshold, sensitivity was 0.676 and PPV was 0.431. These results suggest that the model may help identify high-risk patients for further clinical review. However, more validation is needed before clinical use.

**Limitations**

This study has several limitations. First, the datasets came from the same challenge, so the results may not generalize to other hospitals. Second, missing measurements may affect prediction accuracy. In particular, 2,403 measurements in Set C had missing parameter names and were excluded. Third, model performance varied across patient subgroups. These limitations should be considered when interpreting the results.

**Information Leakage**

To reduce data leakage, the model used measurements from the first 48 hours of ICU admission. The final model and preprocessing pipeline were fitted on Set A and applied to Set B and Set C without refitting. This reduces the risk of information leakage during external evaluation. However, the model should not be interpreted as an admission-time prediction model. The timing of measurements and outcomes should also be checked before clinical application.

**Causal Interpretation**

This study focused on prediction rather than causal inference. Important predictors may be associated with mortality, but this does not mean they directly cause mortality. Therefore, feature importance and partial dependence plots should not be interpreted as evidence of causal relationships.
